# Problem Statement

## Business Context

As organizations grow and scale, they are often inundated with large volumes of data, reports, and documents that contain critical information for decision-making. In real-world business settings, such as venture capital firms like Andreesen Horowitz, business analysts are required to sift through large datasets, research papers, or reports to extract relevant information that impacts strategic decisions.

For instance, consider that you've just joined Andreesen Horowitz, a renowned venture capital firm, and you are tasked with analyzing a dense report like the Harvard Business Review's **"How Apple is Organized for Innovation."** Going through the report manually can be extremely time-consuming as the size and complexity of these report increases. However, by using **Semantic Search** and **Retrieval-Augmented Generation (RAG)** models, you can significantly streamline this process.

Imagine having the capability to directly ask questions like, “How does Apple structure its teams for innovation?” and get immediate, relevant answers drawn from the report. This ability to extract and organize specific insights quickly and accurately enables you to focus on higher-level analysis and decision-making, rather than being bogged down by information retrieval.

## Objective

The goal is to develop a RAG application that helps business analysts efficiently extract key insights from extensive reports, such as “How Apple is Organized for Innovation.”

Specifically, the system aims to:

- Answer user queries by retrieving relevant content directly from lengthy documents.

- Support natural-language interaction without requiring a full manual read-through.

- Act as an intelligent assistant that streamlines the report analysis process.

Through this solution, analysts can save time, improve productivity, and make faster, more informed strategic decisions

## Data Description

**How Apple is Organized for Innovation** - An article of 11 pages in pdf format

## Installing and Importing Necessary Libraries and Dependencies

In [2]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30

**Comments**

**Issue Encountered**

While executing the following line of code:

relevant_document_chunks = retriever.get_relevant_documents(query=user_input, k=k)

I encountered the following error:

AttributeError: module 'langchain' has no attribute 'debug'
**Root Cause**

This error occurs when the installed versions of langchain and langchain-core are incompatible. The langchain-core package expects APIs that are not available in the installed version of langchain, resulting in the AttributeError.

**Resolution**

To resolve the issue, I installed compatible versions of both langchain and langchain-core within the same installation cell. After the installation completed, I restarted the runtime to ensure the updated package versions were loaded correctly.

Once the runtime was restarted, the error was resolved, and the code executed successfully.

In [4]:
!pip install -q -U \
    "langchain>=0.3,<0.4" \
    "langchain-core>=0.3,<0.4" \
    "langchain-community>=0.3,<0.4" \
    "langchain-openai>=0.2,<0.3" \
    "langchain-text-splitters>=0.3,<0.4"


**Comments**: Testing to check if the debug attribute works or not

In [5]:
import langchain, langchain_core
print(langchain.__version__, langchain_core.__version__)
print(hasattr(langchain, "debug"))

0.3.30 0.3.86
True


In [6]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import PyMuPDFLoader                            # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain.embeddings.openai import OpenAIEmbeddings                        # Create vector embeddings using OpenAI's models  # type: ignore
from langchain.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore


from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation

OpenAI API Calling

Mounting the Drive

In [7]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# Load the JSON file and extract values
file_name = '/content/drive/MyDrive/Colab Notebooks/config.json'                                                     # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    API_KEY = config.get("OPENAI_API_KEY")                                      # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()                                                               # Create an instance of the OpenAI client

## Question Answering using LLM

#### Downloading and Loading the model

In [9]:
# Define a function to get a response
def response(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                     # Specify the model to use
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply                                                        # Execute the function with the prompts

### Question 1: Who are the authors of this article and who published this article?

In [11]:
question_1 = "Who is the author of this article and who published this article?"
base_prompt_response_1 = response(question_1)
base_prompt_response_1

"I'm sorry, but I cannot access external articles or content, including their authors or publishers. If you can provide me with the text or details from the article, I may be able to help you analyze it or find relevant information."

**Comments**: I am convinced with the response as at this point there is no proper information about the article and its details and no prompt engnineering techniques applied to guide the model to respond in a more structured way.


### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [13]:
question_2 = "List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines."
base_prompt_response_2 = response(question_2)
base_prompt_response_2

'- **Vision**: Effective leaders possess a clear vision of their goals and objectives, enabling them to inspire and align their team towards a common purpose.\n\n- **Empathy**: Great leaders demonstrate empathy by understanding and valuing the perspectives and emotions of their team members, fostering a supportive and inclusive environment.\n\n- **Integrity**: Leaders with integrity uphold ethical standards and are honest in their actions and decisions, earning the trust and respect of their team and stakeholders.'

**Comments**: At this point, I find the response acceptable because the model was not supplied with enough information about the article. Therefore, it produced a general answer using its pretrained knowledge instead of extracting information from the specific article.

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [14]:
question_3 = "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"
base_prompt_response_3 = response(question_3)
base_prompt_response_3

"While I don't have access to specific articles, I can provide a general overview of how Apple's approach to leadership has historically led to successful innovations. Here are a few examples that highlight their leadership style and its impact on innovation:\n\n1. **Focus on Design and User Experience**: Under Steve Jobs, Apple emphasized the importance of design and user experience. This approach led to the development of products like the iPod and the iPhone, which combined aesthetic appeal with intuitive functionality. The leadership's insistence on high standards in design helped differentiate Apple products in a crowded market.\n\n2. **Secrecy and Surprise**: Apple's leadership has often relied on a strategy of secrecy around product development. This creates a sense of anticipation and excitement among consumers. The launch of the original iPhone in 2007 is a prime example, where the lack of information prior to the announcement led to a significant buzz and immediate consumer i

**Comments**: The response appears to be reasonably relevant because the question references Apple. Based on the available information, the model seems to have generated its answer using its existing knowledge of Apple rather than extracting details from a specific article. Since the article content was not provided, the response should be considered a general AI-generated explanation and not a fact verified from the referenced article.

**Observations**:
*  Based on my experiments, I noticed that the model was able to answer general questions reasonably well using its pretrained knowledge. However, the responses were broad and lacked information that was specific to the article because no additional context was provided.
*  The answers were relevant to the questions asked, but they were not grounded in the source document. As a result, there was no way to verify whether the generated information accurately reflected the content of the article.
*  These initial results highlight that using an LLM without prompt engineering is suitable for general-purpose questions. However, for tasks that require document-specific or fact-based responses, additional techniques such as prompt engineering and Retrieval-Augmented Generation (RAG) are necessary to improve accuracy, consistency, and reliability.

## Question Answering using LLM with Prompt Engineering

Define a function to generate a response from LLM

In [15]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                    # Specify the model to use (GPT-4o in this case)
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

Define the Sytstem prompt that aligns with the business requirement

In [16]:
system_prompt = """
You are an AI assistant, you are specialized in analyzing articles related to IT innovations in software, hardware, and services content.
Please rely on the factual correctness while answering the questions. If the query requires specific reference materials beyond the IT innovations knowledge,
acknowledge the limitaion rather than speculating
"""

### Question 1: Who are the authors of this article and who published this article?

In [17]:
response_with_prompt_eng_1 = response(system_prompt, question_1)
response_with_prompt_eng_1

"I'm sorry, but I don't have access to external articles or their specific details, including authorship or publication information. If you provide me with the content or main points of the article, I can help analyze it or discuss its themes related to IT innovations."

**Comments**: The response is consistent with the instructions provided in the system prompt.  Since the model was explicitly directed to avoid Hallucinations and rely only on factual information, it has completely stopped from generating unsupported details. The response aligns with expected behaviour.

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [18]:
response_with_prompt_eng_2 = response(system_prompt, question_2)
response_with_prompt_eng_2

"Here are three important leadership characteristics:\n\n- **Vision**: A strong leader possesses a clear vision for the future, enabling them to inspire and guide their team towards achieving long-term goals. This foresight helps in aligning the team's efforts with the organization's objectives.\n\n- **Empathy**: Effective leaders demonstrate empathy by understanding and valuing the feelings and perspectives of their team members. This fosters a positive work environment and strengthens relationships, leading to improved collaboration and morale.\n\n- **Decisiveness**: A good leader is decisive, capable of making informed decisions promptly even in uncertain situations. This ability instills confidence in the team, ensuring that progress continues and challenges are addressed swiftly."

**Comments**: The response is generic and closely resembles the output generated without prompt engineering. This behavior is expected because leadership qualities are generally universal and apply across various domains, including healthcare, information technology, education, and other industries. As a result, the model provides broadly applicable leadership characteristics rather than domain-specific ones.

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [19]:
response_with_prompt_eng_3 = response(system_prompt, question_3)
response_with_prompt_eng_3

"I’m unable to access specific articles or their content directly. However, I can provide general examples of how Apple's approach to leadership has historically led to successful innovations.\n\n1. **Visionary Leadership**: Under Steve Jobs, Apple emphasized a clear vision that combined technology with aesthetics. This approach led to the creation of iconic products like the iPhone and iPad, which revolutionized their respective markets.\n\n2. **Focus on User Experience**: Apple's leadership has consistently prioritized user experience, leading to innovations like the App Store, which not only transformed software distribution but also created an entirely new ecosystem for app developers.\n\n3. **Integrated Hardware and Software**: Apple's leadership in tightly integrating hardware and software has led to innovations such as the M1 chip, which allows for greater performance and efficiency in devices like the MacBook.\n\n4. **Brand Loyalty and Marketing**: Apple's approach to marketing

**Comments**: The response is generic and closely resembles the output generated with prompt engineering.

**Observations** :
*  From the initial results, I observed that the LLM was able to answer the questions using its pretrained knowledge, but the responses were broad and not tailored to the specific article. Without additional context, the model generated generalized answers rather than document-specific ones.
*  I also noticed that the model attempted to provide relevant information even when it did not have access to the article. While some responses appeared reasonable, there was no guarantee that they accurately represented the content of the source document.
*  This experiment helped me understand that an LLM on its own is suitable for answering general knowledge questions. However, when the objective is to extract accurate information from a particular document, techniques such as prompt engineering and RAG are required to improve response quality and reliability.

## Data Preparation for RAG

### Loading the Data

In [20]:
# Set the path to the PDF file
manual_pdf_path = "/content/drive/MyDrive/Colab Notebooks/HBR_How_Apple_Is_Organized_For_Innovation.pdf"                       # Path to the medical diagnosis manual PDF

# Load the PDF using LangChain's PyPDFLoader
pdf_loader = PyMuPDFLoader(manual_pdf_path)                                     # Initialize the PDF loader with the file path

# Extract content from the PDF
manual = pdf_loader.load()                                                      # Load and extract text from all pages of the PDF

### Data Overview

In [21]:
for i, doc in enumerate(manual):
    print(f"--- Page {i} | {len(doc.page_content)} chars ---")
    print(doc.page_content[:600])
    print()

--- Page 0 | 353 chars ---
REPRINT R2006F
PUBLISHED IN HBR
NOVEMBER–DECEMBER 2020
ARTICLE
ORGANIZATIONAL CULTURE
How Apple Is 
Organized  
for Innovation
It’s about experts leading experts. 
by Joel M. Podolny and Morten T. Hansen
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.

--- Page 1 | 198 chars ---
2
Harvard Business Review
November–December 2020
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.

--- Page 2 | 495 chars ---
PHOTOGRAPHER MIKAEL JANSSON
How Apple Is 
Organized 
for Innovation
It’s about experts 
leading experts.
ORGANIZATIONAL 
CULTURE
Joel M. 
Podolny
Dean, Apple 
University
Morten T. 
Hansen
Faculty, Apple 
University
AUTHORS
FOR ARTICLE REPRINTS CALL 800-988-0886 OR 617-783-7500, OR VISIT HBR.ORG
Harvard Business Review
November–December 2020  3
This article is ma

**Testing while debugging**

**Comments**: I added this code here to check my data is properly loaded or not and also to see if it can capture the author and publisher's details

### Data Chunking

Chunk the PDF into Manageable Text Sections Using a Token-Based Splitter

In [22]:
# Initialize a text splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',                                                # Encoding used by popular LLMs
    chunk_size=256,                                                             # Each chunk will have up to 256 tokens
    chunk_overlap=20                                                            # 20 tokens will overlap between consecutive chunks (for context continuity)
)

Split the Loaded PDF into Chunks for Further Processing

In [25]:
# Use the text splitter to divide the PDF content into smaller chunks
document_chunks = pdf_loader.load_and_split(text_splitter)

### Generate Vector Embeddings for Text Chunks Using OpenAI

In [26]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model                     # Returns a list of chunked documents

Dimension of the embedding vector  1536


### Vector Database

Setup Vector Store Directory

In [27]:
# Creating a folder for saving the vector DB so it persists between runs
out_dir = 'AppleInnovations_db'                                                          # Directory to store the persistent vector database

# Create the directory if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)                                                        # Make directory to save vector store files

Create Vector Store from Documents

In [28]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

Load Vector Store

In [29]:
# Reloading the vector store from disk without recomputing embeddings
vectorstore = Chroma(
    persist_directory=out_dir,                                                  # Load existing vector DB files
    embedding_function=embedding_model                                          # Use the same embedding function for queries
)

/tmp/ipykernel_896/4264619131.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


Explore Vector Store and Perform Searches

In [30]:
# Inspect the embedding function in use
vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7c32a9702fd0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7c32a9703750>, model='text-embedding-ada-002', deployment='text-embedding-ada-002', openai_api_version='', openai_api_base='https://aibe.mygreatlearning.com/openai/v1', openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key='gl-U2FsdGVkX18L5UirQsZlYI1XkQ3VoJLljWg9bhvoowctTiH1DXkXR+ycP4fnxsg/', openai_organization=None, allowed_special=set(), disallowed_special='all', chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None)

In [31]:
# Search for top 3 most relevant chunks for the query
vectorstore.similarity_search(
    "What are the challanges posed by organizational growth on VPs and Directors?",
    k=3
)

[Document(metadata={'creator': 'Adobe InDesign 14.0 (Macintosh)', 'total_pages': 11, 'file_path': '/content/drive/MyDrive/Colab Notebooks/HBR_How_Apple_Is_Organized_For_Innovation.pdf', 'subject': '', 'creationDate': "D:20201005141842-04'00'", 'encryption': 'Standard V2 R3 128-bit RC4', 'creationdate': '2020-10-05T14:18:42-04:00', 'modDate': 'D:20201201183749Z', 'moddate': '2020-12-01T18:37:49+00:00', 'page': 9, 'format': 'PDF 1.6', 'title': '', 'producer': 'Adobe PDF Library 15.0 (via http://bfo.com/products/pdf?version=2.23.5-r33279)', 'author': '', 'trapped': '', 'keywords': '', 'source': '/content/drive/MyDrive/Colab Notebooks/HBR_How_Apple_Is_Organized_For_Innovation.pdf'}, page_content='As the company has grown, entering new markets and \nmoving into new technologies, its functional structure and \nleadership model have had to evolve. Deciding how to orga-\nnize areas of expertise to best enable collaboration and rapid \ndecision-\u200bmaking has been an important responsibility 

**Comments** : The test produced the expected outcome. The response was generated using information retrieved from the external document and closely matched the relevant content. By searching the three most relevant document chunks (k=3), the retrieval mechanism successfully located the appropriate sections of the article. This confirms that the vector store supplied relevant contextual information, allowing the LLM to produce a response that was accurate and grounded in the source document.

### Retriever

Retrieval and Response Generation using Vector Search

Convert Vector Store into a Retriever and Retrieve Relevant Documents

In [32]:
# Wraping the vector store into a retriever object to fetch the most relevant documents for a given query using similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',                                                   # Use similarity search (based on vector distance)
    search_kwargs={'k': 3}                                                      # Retrieve top 3 most relevant documents
)

System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

1. The system message describing the assistant's role.
2. A user message template including context and the question.

In [33]:
# Define the system prompt for the model
qna_system_message = """
You are an AI assistant designed to support and analyze the IT industries innovations in hardware, software and services. Your task is to provide evidence-based, concise, and relevant information based on the context provided.


User input will include the necessary context for you to answer their questions. This context will begin with the token: ###Context. The context contains references to specific portions of research articles relevant to the query, along with their source details.


When crafting your response:
1. Use only the provided context to answer the question.
2. If the answer is found in the context, respond with concise and actionable insights.
3. Include the source reference with the page number, journal name, or publication, as provided in the context.
4. If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."


Please adhere to the following response guidelines:
- Provide clear, direct answers using only the given context.
- Do not include any additional information outside of the context.
- Avoid rephrasing or summarizing the context unless explicitly relevant to the question.



Here is an example of how to structure your response:


Answer:
[answer based on context]


Source:
[Source details with page or section]
"""

In [34]:
# Define the user message template
qna_user_message_template = """
###Context
Here are some excerpts from Apple organizational culture and their sources that are relevant to the question mentioned below:
{context}

###Question
{question}
"""

### Response Function

In [35]:
def generate_rag_response(user_input,k=3,max_tokens=1000,temperature=0.75,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question 1: Who are the authors of this article and who published this article?

In [36]:
response_with_rag_1 = generate_rag_response(question_1, k=6)

response_with_rag_1

/tmp/ipykernel_896/3563447094.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)


'Answer:\nThe authors of the article are Joel M. Podolny and Morten T. Hansen. It was published by Harvard Business Review.\n\nSource:\nHarvard Business Review, November–December 2020.'

**Comments**: The response met the expected outcome. When the retrieval parameter (k) was not specified or was set to a value lower than 6, the model was unable to retrieve the document chunk containing the author's name. After increasing the value to k=6, the retrieval process successfully identified the relevant chunk that included the required information. This demonstrates the importance of selecting an appropriate k value in a RAG pipeline, as retrieving a larger number of relevant document chunks can improve the likelihood of capturing the necessary context and producing a more accurate and complete response.

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [37]:
response_with_rag_2 = generate_rag_response(question_2)
response_with_rag_2

"Answer:\n- **Deep Expertise**: Leaders are expected to have a profound understanding of their functional areas, enabling meaningful engagement in the work and informed decision-making.\n\n- **Immersion in Details**: Leaders must be deeply involved in the specifics of their functions, allowing them to identify critical issues and guide their teams effectively.\n\n- **Willingness to Collaboratively Debate**: Leaders should advocate for their views while remaining open to others' ideas, fostering a collaborative environment that encourages rigorous discussion and informed decisions.\n\nSource:\n[Organizational Culture, HBR]"

**Comments**: the outcome is relevant, This information is retrieved from page 5, confirming that the retrieval process identified the correct section of the document.

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [38]:
response_with_rag_3 = generate_rag_response(question_3,k=2)
response_with_rag_3

"Answer:\nThe article highlights that Apple's organizational model, which is organized around functions rather than business units, has enabled cross-functional collaboration among leaders who are deeply knowledgeable about the details of their domains. This approach allows decision-makers to have a comprehensive understanding of the technological landscape, fostering innovation. When Steve Jobs returned to Apple, he dismantled the conventional structure with decentralized business units, which he believed stifled innovation, and instead implemented a model that promotes collaboration and shared expertise, leading to significant growth in both employees and revenue.\n\nSource:\nHarvard Business Review, page 4."

**Comments**: The retrieved document chunks were highly relevant to the question and contained the key information needed to generate the response.
The model successfully used the retrieved context to explain Apple's organizational structure and its impact on innovation instead of relying on general knowledge.

## Actionable Insights and Business Recommendations

**Question 1 – Identifying the Author(s) and Publisher**

The RAG implementation was able to identify the author(s) and publisher by retrieving the relevant sections from the uploaded article. During my initial tests, the standalone LLM could not answer this question because the required information was not part of its available context. Once the document was indexed and retrieved through the RAG pipeline, the model generated the correct response using the article itself rather than relying on its pretrained knowledge.

**Question 2 – Leadership Characteristics**

The generated response clearly listed the three leadership characteristics discussed in the article and presented them in the requested bullet-point format. The combination of prompt engineering and RAG produced a response that was concise, well-structured, and closely aligned with the content of the document instead of providing a generic explanation.

**Question 3 – Apple's Leadership and Innovation**

The model successfully extracted examples from the article that demonstrate how Apple's leadership approach has contributed to innovation. Unlike the responses produced without retrieval, the RAG-based response referenced the information available in the document, making the answer more accurate, relevant, and specific to the article.

**Technical Insights**

*  During the experiments, I observed that
prompt engineering improved the structure and consistency of the responses, but it could not compensate for information that was missing from the model's context.
*  Integrating RAG significantly improved response quality because the model was able to retrieve information directly from the uploaded document before generating its answer.
*  The value of the retrieval parameter (k) had a noticeable impact on the results. When a smaller k value was used, some important information, such as the author's details, was not retrieved. Increasing k to 6 enabled the retriever to locate the required document chunk and produce a complete response.
*  The effectiveness of the chatbot depended not only on the LLM but also on the quality of the retrieval process. Accurate embeddings and relevant document chunks played an important role in generating reliable answers.
*  Semantic similarity search proved to be effective because it retrieved information based on meaning rather than exact keyword matches, making the system more flexible when handling different user queries.




**Recommendations**

*  Use prompt engineering to clearly define the model's role, expected behavior, and response format before processing user queries.
*  Experiment with different values of the retrieval parameter (k) to find the best balance between retrieval accuracy and processing efficiency for the given dataset.
*  Select an appropriate chunk size and chunk overlap to ensure that related information is preserved and important context is not split across multiple chunks.
*  Regularly update the document collection so the chatbot can continue to provide responses based on the latest available information.
*  Include document metadata, such as page numbers or section titles, in the final response whenever possible to improve traceability and increase user confidence in the generated answers.
*  Continue evaluating retrieval accuracy and prompt effectiveness, as both components contribute significantly to the overall performance of a RAG-based application.

<p><strong>Key Business Takeaways</strong></p>
<ul>
<li>From this project, I realized that combining an LLM with RAG produces more dependable responses because the answers are generated from the organization's documents instead of relying only on the model's existing knowledge.</li>
<li>I found that prompt engineering plays an important role in guiding the model's behavior. While it cannot create missing information, it helps the chatbot produce responses that are clearer, more consistent, and aligned with the expected format.
</li>
<li>Providing employees with instant access to information through a RAG-based chatbot can significantly reduce the time spent searching through lengthy documents, allowing them to make decisions more efficiently.
</li>
<li>Using semantic search improves the user experience because the chatbot understands the intent behind a question instead of depending only on exact keyword matches, resulting in more relevant answers.</li>
<li>This solution can be applied across different business functions, including customer support, internal knowledge management, technical documentation, policy assistance, and employee onboarding, making it a scalable investment for organizations.
</li>
<li>Overall, the project demonstrates that integrating prompt engineering with RAG leads to a more reliable, efficient, and business-friendly AI solution that supports accurate information retrieval and improves operational productivity.</li>
</ul>
